# AION Chat — Transformer on TPU v5e (Kaggle)

**Phase 2 of 2: Fine-tune Transformer on chat data using pretrained weights.**

Run the pretrain TPU notebook (Phase 1) first to build language foundations.
On first run here, pretrained weights are automatically loaded from the
pretrain Kaggle Dataset (`<user>/aion-transformer-tpu-medium`).

**Target: 20,000 steps across multiple sessions (~4 sessions × 5,000 steps).**

## Before first run
1. `Settings → Accelerator → TPU v5e-8`
2. Add these Kaggle Secrets (`Add-ons → Secrets`):
   - `GITHUB_TOKEN`, `GITHUB_USERNAME` — to clone the repo
   - `KAGGLE_USERNAME`, `KAGGLE_KEY` — for checkpoint Dataset persistence

**Note:** This notebook uses a Transformer model (not Mamba) because
Mamba's custom CUDA kernels are incompatible with TPU/XLA.

In [ ]:
# Cell 1 — Install dependencies, clone repo, configure Kaggle API
import subprocess, sys, os, gc, json
from kaggle_secrets import UserSecretsClient

_sec = UserSecretsClient()
TOKEN           = _sec.get_secret('GITHUB_TOKEN')
GITHUB_USERNAME = _sec.get_secret('GITHUB_USERNAME')
KAGGLE_USERNAME = _sec.get_secret('KAGGLE_USERNAME')
KAGGLE_KEY      = _sec.get_secret('KAGGLE_KEY')

for _name, _val in [('GITHUB_TOKEN', TOKEN), ('GITHUB_USERNAME', GITHUB_USERNAME),
                    ('KAGGLE_USERNAME', KAGGLE_USERNAME), ('KAGGLE_KEY', KAGGLE_KEY)]:
    if not _val:
        raise ValueError(f"Add a Kaggle secret named '{_name}' (Add-ons -> Secrets).")

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)
os.environ['KAGGLE_USERNAME'] = KAGGLE_USERNAME
os.environ['KAGGLE_KEY']      = KAGGLE_KEY

REPO_URL = f'https://{TOKEN}@github.com/{GITHUB_USERNAME}/aion.git'

# torch_xla is pre-installed on Kaggle TPU; only need data/tokenizer deps
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'datasets', 'tokenizers', 'pyyaml', 'tqdm', 'kaggle',
], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'cache', 'purge'], check=False)

if not os.path.exists('/tmp/aion'):
    subprocess.run(['git', 'clone', '--depth=1', '-b', 'main', REPO_URL, '/tmp/aion'], check=True)
    print('Repo cloned (main branch).')
else:
    subprocess.run(['git', '-C', '/tmp/aion', 'pull', '--ff-only'], check=True)
    print('Repo up to date.')

if '/tmp/aion/src' not in sys.path:
    sys.path.insert(0, '/tmp/aion/src')

for _key in list(sys.modules.keys()):
    if _key.startswith('llm_lab'):
        del sys.modules[_key]

# Verify torch_xla is present WITHOUT initializing the TPU runtime.
# IMPORTANT: do NOT call xm.xla_device() (or any XLA op) in this kernel — that would
# claim the single-host TPU here and make the multi-core xmp.spawn in Cell 5 abort
# with 'runtime_metric_aggregator Check failed: reporting_closure_ == nullptr'.
# The training subprocess must be the first process to touch the TPU.
import importlib.util
assert importlib.util.find_spec('torch_xla') is not None, \
    'torch_xla not available — set Settings -> Accelerator -> TPU v5e-8.'
print('torch_xla available (TPU left uninitialized so the multi-core spawn can own it).')
print('Done.')


In [ ]:
# Cell 2 — Restore checkpoints from Kaggle Dataset + check progress
from pathlib import Path
import subprocess, json

# Which pretrained base to fine-tune: 'medium' (110M) or 'large' (~235M). Must match a
# base you actually pretrained with kaggle_pretrain_tpu.ipynb (same MODEL_SIZE there).
MODEL_SIZE = 'medium'

# Chat checkpoint Datasets. medium keeps the rescued '-v2' slug: the original
# 'aion-transformer-tpu-chat' was deleted and Kaggle forbids re-versioning a deleted slug.
_CHAT_SLUGS = {
    'medium': 'aion-transformer-tpu-chat-v2',
    'large':  'aion-transformer-tpu-chat-large',
}
DATASET_SLUG = f'{KAGGLE_USERNAME}/{_CHAT_SLUGS[MODEL_SIZE]}'
PRETRAIN_DATASET_SLUG = f'{KAGGLE_USERNAME}/aion-transformer-tpu-{MODEL_SIZE}'
CKPT_DIR = Path('/tmp/checkpoints')
CKPT_DIR.mkdir(parents=True, exist_ok=True)

res = subprocess.run(
    ['kaggle', 'datasets', 'download', '-d', DATASET_SLUG, '-p', str(CKPT_DIR), '--unzip'],
    capture_output=True, text=True)
DATASET_EXISTS = res.returncode == 0
if DATASET_EXISTS:
    print('Chat checkpoints restored from Kaggle Dataset.')
else:
    print('No chat checkpoint dataset yet (first session) - it will be created after training.')
    _err = (res.stderr or res.stdout).strip().splitlines()
    if _err:
        print('  ', _err[-1])

latest = CKPT_DIR / 'latest.pt'
if latest.exists():
    import torch
    meta = torch.load(latest, map_location='cpu', weights_only=False)
    steps_done = meta.get('step', 0)
    from llm_lab.run_config import budget
    TARGET_STEPS = budget('chat_medium')   # central step budget (repo, not notebook)
    print(f'Progress: {steps_done:,} / {TARGET_STEPS:,} steps ({steps_done/TARGET_STEPS*100:.1f}%)')
    if meta.get('metrics_log'):
        vals = [e for e in meta['metrics_log'] if 'val_loss' in e]
        if vals:
            print(f'Last val_loss: {vals[-1]["val_loss"]:.4f}')
else:
    print('No checkpoint found - this is the first session.')

print(f'Model size: {MODEL_SIZE} | seed base: {PRETRAIN_DATASET_SLUG} | Checkpoint dir: {CKPT_DIR}')


In [ ]:
# Cell 3 — Download chat datasets
import sys, runpy
from pathlib import Path

for _key in list(sys.modules.keys()):
    if _key.startswith('llm_lab'):
        del sys.modules[_key]

RAW_DIR = '/tmp/data/raw'

sys.argv = ['cli', 'download', '--target', RAW_DIR, '--preset', 'chat']
runpy.run_module('llm_lab.cli', run_name='__main__', alter_sys=True)
print('Download complete.')

In [ ]:
# Cell 4 — Merge datasets + tokenizer
# The chat model MUST use the same tokenizer as the pretrained model (same vocab).
import shutil, sys, runpy, json, subprocess
from pathlib import Path

RAW_DIR    = Path('/tmp/data/raw')
DATA_DIR   = Path('/tmp/data')
DATA_DIR.mkdir(parents=True, exist_ok=True)

SEED_PATH      = '/tmp/aion/src/llm_lab/data/instruction_seed.json'
MERGED_PATH    = '/tmp/data/chat_merged.json'
TOKENIZER_PATH = '/tmp/data/tokenizer.json'
CKPT_TOK       = CKPT_DIR / 'tokenizer.json'

# --- Reuse tokenizer: prefer pretrain Dataset (same vocab as model weights) ---
need_tokenizer = True
if CKPT_TOK.exists():
    shutil.copy2(CKPT_TOK, TOKENIZER_PATH)
    print(f'Tokenizer restored from chat checkpoint dir.')
    need_tokenizer = False
else:
    _pretrain_dir = Path('/tmp/pretrain_ckpt')
    _pretrain_dir.mkdir(parents=True, exist_ok=True)
    _r = subprocess.run(
        ['kaggle', 'datasets', 'download', '-d', PRETRAIN_DATASET_SLUG,
         '-f', 'tokenizer.json', '-p', str(_pretrain_dir), '--force'],
        capture_output=True, text=True)
    _pretrain_tok = _pretrain_dir / 'tokenizer.json'
    if _pretrain_tok.exists():
        shutil.copy2(_pretrain_tok, TOKENIZER_PATH)
        shutil.copy2(_pretrain_tok, CKPT_TOK)
        print(f'Tokenizer restored from pretrain Dataset.')
        need_tokenizer = False

# --- Always rebuild chat_merged.json (ephemeral disk) ---
sys.argv = [
    'cli', 'merge-chat',
    '--raw-dir', str(RAW_DIR),
    '--out',     MERGED_PATH,
    '--seed',    SEED_PATH,
    '--max-hh-rlhf', '20000',
]
runpy.run_module('llm_lab.cli', run_name='__main__', alter_sys=True)

# --- Train tokenizer only if not restored ---
if need_tokenizer:
    chat_examples = json.loads(Path(MERGED_PATH).read_text())
    corpus_path = DATA_DIR / 'corpus.txt'
    with open(corpus_path, 'w', encoding='utf-8') as f:
        for ex in chat_examples:
            for msg in ex['messages']:
                f.write(msg['content'].strip() + '\n')
    sys.argv = ['cli', 'tokenizer', '--corpus', str(corpus_path), '--out', TOKENIZER_PATH, '--vocab-size', '16384']
    runpy.run_module('llm_lab.cli', run_name='__main__', alter_sys=True)
    shutil.copy2(TOKENIZER_PATH, CKPT_TOK)
    print('Tokenizer trained and saved to checkpoint dir.')

print('Data ready.')

In [ ]:
# Cell 5 — Train / resume on TPU, then push checkpoints
import sys, runpy, yaml, shutil, os, gc, json
import subprocess
import torch
from pathlib import Path

STEPS_PER_SESSION = 5000
_MODEL_SIZE = globals().get('MODEL_SIZE', 'medium')  # set in Cell 2

# Set True ONCE to discard the existing chat checkpoint and re-seed from the latest
# pretrained base (e.g. after re-pretraining). Set back to False for normal resumes.
FRESH_START = False

for _key in list(sys.modules.keys()):
    if _key.startswith('llm_lab'):
        del sys.modules[_key]

gc.collect()

# --- Optional fresh start: drop the restored chat checkpoint so we re-seed from the base ---
if FRESH_START:
    for _f in ('latest.pt', 'best.pt', 'metrics.json'):
        _p = CKPT_DIR / _f
        if _p.exists():
            _p.unlink()
    print('FRESH_START: cleared chat checkpoint — will re-seed from the pretrained base.')

# --- If pretrained weights exist but no chat checkpoint yet, seed from pretrain ---
latest = CKPT_DIR / 'latest.pt'

if not latest.exists():
    _pretrain_dir = Path('/tmp/pretrain_ckpt')
    _pretrain_dir.mkdir(parents=True, exist_ok=True)
    pretrain_src = None
    for _fname in ('best.pt', 'latest.pt'):
        _r = subprocess.run(
            ['kaggle', 'datasets', 'download', '-d', PRETRAIN_DATASET_SLUG,
             '-f', _fname, '-p', str(_pretrain_dir), '--force'],
            capture_output=True, text=True)
        if (_pretrain_dir / _fname).exists():
            pretrain_src = _pretrain_dir / _fname
            break
    if pretrain_src:
        _ckpt = torch.load(pretrain_src, map_location='cpu', weights_only=False)
        _ckpt['step'] = 0
        _ckpt['optimizer'] = None
        _ckpt['scheduler'] = None
        _ckpt['metrics_log'] = []
        torch.save(_ckpt, latest)
        del _ckpt
        print(f'Seeded chat training from pretrained checkpoint: {pretrain_src.name}')
    else:
        print('No pretrained checkpoint found - training chat model from scratch.')

latest = CKPT_DIR / 'latest.pt'
is_resume = latest.exists()

# Config family follows the base size selected in Cell 2 (dims must match the seed weights).
_CFG = {
    'medium': ('transformer_tpu_chat.yaml', 'transformer_tpu_chat_resume.yaml'),
    'large':  ('transformer_tpu_chat_large.yaml', 'transformer_tpu_chat_large_resume.yaml'),
}[_MODEL_SIZE]
_CFG_DIR = Path('/tmp/aion/src/llm_lab/configs')

if is_resume:
    metrics_path = CKPT_DIR / 'metrics.json'
    if metrics_path.exists():
        _metrics = json.loads(metrics_path.read_text())
        steps_done = max((e.get('step', 0) for e in _metrics), default=0)
    else:
        steps_done = 0
    print(f'Found checkpoint at step {steps_done:,} - resuming with low LR.')
    cfg_path = _CFG_DIR / _CFG[1]
else:
    steps_done = 0
    print('No checkpoint found - starting from scratch with full LR.')
    cfg_path = _CFG_DIR / _CFG[0]

if not cfg_path.exists():
    raise FileNotFoundError(f'Config not found: {cfg_path}')

cfg = yaml.safe_load(cfg_path.read_text())
print(f'Using config: {cfg_path.name}')

cfg['dataset_type']     = 'chat'
cfg['instruction_data'] = '/tmp/data/chat_merged.json'
cfg['train_path']       = ''
cfg['val_path']         = ''
cfg['tokenizer_path']   = '/tmp/data/tokenizer.json'
cfg['checkpoint_dir']   = str(CKPT_DIR)
cfg['max_steps']        = steps_done + STEPS_PER_SESSION

run_cfg_path = '/tmp/run_chat_tpu.yaml'
Path(run_cfg_path).write_text(yaml.dump(cfg))
shutil.copy2(run_cfg_path, CKPT_DIR / 'run.yaml')

print(f'Steps this session: {STEPS_PER_SESSION} (step {steps_done:,} -> {cfg["max_steps"]:,})')
print(f'lr={cfg["lr"]:.1e}, warmup={cfg["warmup_steps"]}, seq_len={cfg["seq_len"]}')
print(f'batch_size={cfg["batch_size"]}, model: d_model={cfg["d_model"]}, layers={cfg["n_layers"]}')
print(f'Checkpoints -> {CKPT_DIR}')
print()

def _push_checkpoints():
    push_dir = Path('/tmp/ckpt_push')
    if push_dir.exists():
        shutil.rmtree(push_dir)
    push_dir.mkdir(parents=True, exist_ok=True)
    for name in ('latest.pt', 'best.pt', 'metrics.json', 'run.yaml', 'tokenizer.json'):
        src = CKPT_DIR / name
        if src.exists():
            shutil.copy2(src, push_dir / name)
    if not (push_dir / 'latest.pt').exists():
        print('No latest.pt to push.')
        return
    meta = {
        'title': f'AION Transformer TPU Chat Checkpoints ({_MODEL_SIZE})',
        'id': DATASET_SLUG,
        'licenses': [{'name': 'CC0-1.0'}],
    }
    (push_dir / 'dataset-metadata.json').write_text(json.dumps(meta))
    try:
        _steps = json.loads((CKPT_DIR / 'metrics.json').read_text())
        _last = max((e.get('step', 0) for e in _steps), default=0)
    except Exception:
        _last = 0
    print(f'Pushing checkpoints to {DATASET_SLUG} (step {_last})...')
    # Version an existing dataset; fall back to create when it doesn't exist OR the slug
    # is forbidden (e.g. it was deleted -> Kaggle returns 403 on version, not 404).
    version_cmd = ['kaggle', 'datasets', 'version', '-p', str(push_dir),
                   '-m', f'step {_last}', '--dir-mode', 'zip']
    create_cmd = ['kaggle', 'datasets', 'create', '-p', str(push_dir), '--dir-mode', 'zip']
    r = subprocess.run(version_cmd, capture_output=True, text=True)
    _out = (r.stdout or '') + (r.stderr or '')
    if r.returncode != 0 and any(s in _out.lower()
                                 for s in ('not found', "doesn't exist", 'does not exist',
                                           '404', '403', 'forbidden')):
        r = subprocess.run(create_cmd, capture_output=True, text=True)
        _out = (r.stdout or '') + (r.stderr or '')
    print(_out.strip())

try:
    if '/tmp/aion/src' not in sys.path:
        sys.path.insert(0, '/tmp/aion/src')
    sys.argv = ['cli', 'train', '--config', run_cfg_path]
    runpy.run_module('llm_lab.cli', run_name='__main__', alter_sys=True)
finally:
    _push_checkpoints()
    for _key in list(sys.modules.keys()):
        if _key.startswith('llm_lab'):
            del sys.modules[_key]
    gc.collect()

In [ ]:
# Cell 6 — Test the model
import torch
from pathlib import Path
from tokenizers import Tokenizer

sys.path.insert(0, '/tmp/aion/src')
from llm_lab.training.config import TrainConfig
from llm_lab.training.model_factory import build_model

device = torch.device('cpu')  # inference on CPU for simplicity
tokenizer = Tokenizer.from_file('/tmp/data/tokenizer.json')
cfg = TrainConfig.load(Path('/tmp/run_chat_tpu.yaml'))

model = build_model(cfg).to(device)

best_ckpt = CKPT_DIR / 'best.pt'
latest_ckpt = CKPT_DIR / 'latest.pt'
ckpt_path = best_ckpt if best_ckpt.exists() else latest_ckpt
if not ckpt_path.exists():
    raise FileNotFoundError(f'No checkpoint found in {CKPT_DIR}')

ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model'], strict=False)
model.eval()
print(f'Loaded {ckpt_path.name} from step {ckpt["step"]:,}')

def chat(user_message: str, max_new_tokens: int = 200, temperature: float = 0.8) -> str:
    prompt = f'<|user|>{user_message}<|end|>\n<|assistant|>'
    input_ids = torch.tensor([tokenizer.encode(prompt).ids], dtype=torch.long).to(device)
    end_id = tokenizer.encode('<|end|>').ids[0]
    with torch.no_grad():
        for _ in range(max_new_tokens):
            logits = model(input_ids)
            next_logits = logits[:, -1, :] / temperature
            probs = torch.softmax(next_logits, dim=-1)
            next_id = torch.multinomial(probs, 1)
            input_ids = torch.cat([input_ids, next_id], dim=1)
            if next_id.item() == end_id:
                break
    generated = tokenizer.decode(input_ids[0].tolist())
    marker = '<|assistant|>'
    if marker in generated:
        generated = generated.split(marker, 1)[1].replace('<|end|>', '').strip()
    return generated

print('User: What is machine learning?')
print('Assistant:', chat('What is machine learning?'))